## Sparse Mixture of Experts (MoE) — Core Math

### Notation

- $x \in \mathbb{R}^d$: input token representation
- $n$: number of experts
- $k$: number of experts selected per token (top-$k$)
- $E_i(\cdot)$: expert network $i$
- $W_g \in \mathbb{R}^{d \times n}$: router (gating) weight matrix

---

### Router (Gating Network)

1. **Compute router logits:**

   $$l = x \cdot W_g$$
   *(shape: $[n]$)*

2. **Select the top-k logits:**

   $$l_{\text{topk}} = \text{TopK}(l, k)$$

   All non–top-$k$ values are discarded (set to $-\infty$ before softmax or masked out).

3. **Normalize selected logits with softmax:**

   $$g = \text{Softmax}(l_{\text{topk}})$$
   *(shape: $[k]$)*

   Constraint: $\sum_{j=1}^k g_j = 1$

   *Only the $k$ selected experts receive non-zero probability.*

---

### Experts

Each expert maps input to output space:

$$E_i : \mathbb{R}^d \to \mathbb{R}^d$$

Example (MLP / SwiGLU-style):

$$E_i(x) = W_{2,i}( \sigma( W_{1,i} x ) )$$

*Each expert has its own unique parameters.*

---

### MoE Output (Single Token)

Let $e_j$ be the index of the $j$-th selected expert.

$$y = \sum_{j=1}^k g_j \cdot E_{e_j}(x)$$

*Only the selected experts are evaluated (sparse execution).*

---

### Batch Formulation

For batch index $b = 1 \dots B$:

$$y_b = \sum_{j=1}^k g_{b,j} \cdot E_{e_{b,j}}(x_b)$$

*Routing is performed independently per token.*

---

### Key Properties

- **Sparse computation**: only $k \ll n$ experts evaluated per token.
- **Conditional computation**: expert choice depends on input $x$.
- **Gating weights** are differentiable.
- **Expert indices** (TopK selection) are discrete.

---

### Typical Hyperparameters (Mixtral-style)

- Number of experts: $n = 8$
- Experts per token: $k = 2$
- MoE replaces the Feed-Forward Network (FFN) block in each Transformer layer.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SwiGLUExpert(nn.Module):
    """
    SwiGLU FFN Expert as used in Mixtral / LLaMA.
    """
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        # 1. The Gate Projection (w1)
        # Determines which features get passed through the activation
        # Internal "Gate" - controls feature activation, NOT expert selection
        self.gate_proj = nn.Linear(input_dim, hidden_dim, bias=False)
        
        # 2. The Up Projection (w3)
        # A linear transformation of the input, parallel to the gate
        self.up_proj = nn.Linear(input_dim, hidden_dim, bias=False)
        
        # 3. The Down Projection (w2)
        # Projects the combined result back to the original dimension
        self.down_proj = nn.Linear(hidden_dim, input_dim, bias=False)

    def forward(self, x):
        # x shape: [batch_size, seq_len, input_dim]
        # SwiGLU activation: (SiLU(Gate) * Up) -> Down
        
        # Step 1: Compute the Gate (with SiLU activation)
        gate = F.silu(self.gate_proj(x))
        
        # Step 2: Compute the Up projection (Linear, no activation)
        up = self.up_proj(x)
        
        # Step 3: Element-wise multiplication (The "GLU" part)
        fused = gate * up
        
        # Step 4: Project back down to original size
        output = self.down_proj(fused)
        
        return output


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SwiGLUExpert(nn.Module):
    """
    SwiGLU FFN Expert as used in Mixtral / LLaMA.
    """
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        # 1. The Gate Projection (w1)
        # Determines which features get passed through the activation
        # Internal "Gate" - controls feature activation, NOT expert selection
        self.gate_proj = nn.Linear(input_dim, hidden_dim, bias=False)
        
        # 2. The Up Projection (w3)
        # A linear transformation of the input, parallel to the gate
        self.up_proj = nn.Linear(input_dim, hidden_dim, bias=False)
        
        # 3. The Down Projection (w2)
        # Projects the combined result back to the original dimension
        self.down_proj = nn.Linear(hidden_dim, input_dim, bias=False)

    def forward(self, x):
        # x shape: [batch_size, seq_len, input_dim]
        # SwiGLU activation: (SiLU(Gate) * Up) -> Down
        
        # Step 1: Compute the Gate (with SiLU activation)
        gate = F.silu(self.gate_proj(x))
        
        # Step 2: Compute the Up projection (Linear, no activation)
        up = self.up_proj(x)
        
        # Step 3: Element-wise multiplication (The "GLU" part)
        fused = gate * up
        
        # Step 4: Project back down to original size
        output = self.down_proj(fused)
        
        return output


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SwiGLUExpert(nn.Module):
    """
    SwiGLU FFN Expert as used in Mixtral / LLaMA.
    """
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        # 1. The Gate Projection (w1)
        # Determines which features get passed through the activation
        # Internal "Gate" - controls feature activation, NOT expert selection
        self.gate_proj = nn.Linear(input_dim, hidden_dim, bias=False)
        
        # 2. The Up Projection (w3)
        # A linear transformation of the input, parallel to the gate
        self.up_proj = nn.Linear(input_dim, hidden_dim, bias=False)
        
        # 3. The Down Projection (w2)
        # Projects the combined result back to the original dimension
        self.down_proj = nn.Linear(hidden_dim, input_dim, bias=False)

    def forward(self, x):
        # x shape: [batch_size, seq_len, input_dim]
        # SwiGLU activation: (SiLU(Gate) * Up) -> Down
        
        # Step 1: Compute the Gate (with SiLU activation)
        gate = F.silu(self.gate_proj(x))
        
        # Step 2: Compute the Up projection (Linear, no activation)
        up = self.up_proj(x)
        
        # Step 3: Element-wise multiplication (The "GLU" part)
        fused = gate * up
        
        # Step 4: Project back down to original size
        output = self.down_proj(fused)
        
        return output


 The SiLU (Sigmoid Linear Unit) is a specific activation function that combines linear behavior with the sigmoid function. It is also commonly known as Swish (specifically Swish-1).

Here is the mathematical breakdown.

The Equation
$$ \text{SiLU}(x) = x \cdot \sigma(x) $$

Where $\sigma(x)$ is the standard logistic sigmoid function:

$$ \sigma(x) = \frac{1}{1 + e^{-x}} $$

So, the full expanded equation is:

$$ \text{SiLU}(x) = \frac{x}{1 + e^{-x}} $$

How it Works (The "Soft" Gate)
To understand why this is used in the SwiGLU expert, look at the behavior of the two components:

$x$ (The Input): This is the raw value coming from the linear layer.
$\sigma(x)$ (The Gate): The sigmoid function squashes the input between 0 and 1.
The multiplication $x \cdot \sigma(x)$ creates a self-gating mechanism:

If $x$ is a large positive number: $\sigma(x) \approx 1$. Therefore, $\text{SiLU}(x) \approx x \cdot 1 = x$. Result: The signal passes through almost unchanged (like ReLU).

If $x$ is a large negative number: $\sigma(x) \approx 0$. Therefore, $\text{SiLU}(x) \approx x \cdot 0 = 0$. Result: The signal is suppressed (like ReLU).

If $x$ is near zero (The "Magic" Zone): This is where SiLU differs from ReLU.

ReLU has a sharp "kink" at 0 (non-differentiable).
SiLU is smooth and curvy.
SiLU is non-monotonic: It actually dips slightly below zero (minimum value $\approx -0.278$ at $x \approx -1.28$) before curving back up to zero. This allows the network to preserve a small amount of negative information, which helps with gradient flow.

In [3]:
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])

In [4]:
output = F.silu(x)

In [5]:
output

tensor([-0.2384, -0.2689,  0.0000,  0.7311,  1.7616])

In [6]:
F.relu(x)

tensor([0., 0., 0., 1., 2.])

The Formula
$$ \text{ReLU}(x) = \max(0, x) $$

It can also be written as a piecewise function:

$$ \text{ReLU}(x) = \begin{cases} x & \text{if } x > 0 \ 0 & \text{if } x \le 0 \end{cases} $$



### Router (Top-k softmax)
*Formula*

$$Softmax(TopK(xWg​))$$



In [14]:
class TopKRouter(nn.Module):
    def __init__(self, d_model, num_experts, k):
        super().__init__()
        self.k = k
        self.linear = nn.Linear(d_model, num_experts, bias=False)

    def forward(self, x):
        # x: [batch, d_model]
        logits = self.linear(x)                     # [batch, num_experts]
        topk_logits, topk_idx = logits.topk(self.k, dim=-1)

        gates = F.softmax(topk_logits, dim=-1)     # [batch, k]
        return topk_idx, gates


## putting together

In [66]:
import torch
import torch.nn as nn

class MoELayer(nn.Module):
    def __init__(self, d_model, d_hidden, num_experts, k):
        super().__init__()
        self.router = TopKRouter(d_model, num_experts, k)
        self.experts = nn.ModuleList(
            [SwiGLUExpert(d_model, d_hidden) for _ in range(num_experts)]
        )

    def forward(self, x):
        # x: [batch, d_model]
        topk_idx, gates = self.router(x)  # topk_idx: [batch, k], gates: [batch, k]
        
        output = torch.zeros_like(x)

        # Iterate over all experts. 
        # This loop is static (fixed range), avoiding the dynamic .unique() sync.
        for i, expert in enumerate(self.experts):
            # Create a boolean mask for all tokens assigned to this expert 'i'
            # This checks all 'k' slots at once.
            mask = (topk_idx == i) # [batch, k]
            
            # Optimization: Skip expert if no tokens are assigned to it
            if mask.any():
                # Get the batch indices (which sample) and k indices (which rank)
                batch_indices, k_indices = torch.where(mask)
                
                # Gather inputs for this expert
                # x[batch_indices] shape: [num_selected_tokens, d_model]
                expert_input = x[batch_indices]
                
                # Forward pass through the specific expert
                expert_output = expert(expert_input)
                
                # Apply the gating weight
                # gates[batch_indices, k_indices] shape: [num_selected_tokens]
                gate_weight = gates[batch_indices, k_indices].unsqueeze(-1)
                weighted_expert_output = expert_output * gate_weight
                
                # Scatter-add the results back to the output tensor.
                # index_add_ is efficient and handles cases where a single sample 
                # might send to the same expert multiple times (rare, but possible).
                output.index_add_(0, batch_indices, weighted_expert_output)
                
        return output


In [67]:
d_model = 384 
d_hidden = 384*4
num_experts = 8 
k = 2

moe = MoELayer(d_model, d_hidden, num_experts, k) 

In [70]:
moe

MoELayer(
  (router): TopKRouter(
    (linear): Linear(in_features=384, out_features=8, bias=False)
  )
  (experts): ModuleList(
    (0-7): 8 x SwiGLUExpert(
      (gate_proj): Linear(in_features=384, out_features=1536, bias=False)
      (up_proj): Linear(in_features=384, out_features=1536, bias=False)
      (down_proj): Linear(in_features=1536, out_features=384, bias=False)
    )
  )
)

In [72]:
torch.manual_seed(204)
my_data = torch.rand(16,384)

moe(my_data)

tensor([[ 0.0026,  0.0248,  0.0125,  ..., -0.0049, -0.0193, -0.0021],
        [ 0.0233, -0.0281,  0.0028,  ..., -0.0260, -0.0146, -0.0420],
        [ 0.0090, -0.0245, -0.0083,  ..., -0.0791,  0.0204,  0.0084],
        ...,
        [ 0.0020,  0.0120,  0.0069,  ..., -0.0212, -0.0186,  0.0085],
        [ 0.0023,  0.0080,  0.0030,  ...,  0.0164,  0.0069,  0.0233],
        [ 0.0205, -0.0031, -0.0138,  ..., -0.0086, -0.0193, -0.0185]],
       grad_fn=<IndexAddBackward0>)